# REAL DATA 2023Q4 Tuner

### Imports

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import optuna
import optuna.visualization.matplotlib as ovm
from statsmodels.tools.sm_exceptions import ConvergenceWarning, ValueWarning

# --- Clean Console Output ---
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=ValueWarning, module="statsmodels.tsa.base.tsa_model")
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

sys.path.append(os.path.abspath('../scripts'))

from spread import SPREAD
from engine import ENGINE
from backtester import BACKTESTER

### Control panel

In [ ]:
ACTIVE_PAIR = "AUDUSD_NZDUSD"  

# Dictionary mapping pairs to their specific data files and active liquidity hours
PAIR_CONFIG = {
    "AUDUSD_NZDUSD": {"sym_a": "audusd", "sym_b": "nzdusd", "hours": (0, 9)},   # Pacific/Asian Session
    "EURNOK_EURSEK": {"sym_a": "eurnok", "sym_b": "eursek", "hours": (6, 15)},  # European Session
    "GBPUSD_EURUSD": {"sym_a": "gbpusd", "sym_b": "eurusd", "hours": (7, 16)}   # London/NY Overlap
}

cfg = PAIR_CONFIG[ACTIVE_PAIR]
print(f"=== INITIALIZED PIPELINE FOR: {ACTIVE_PAIR} ===")

### Data

In [ ]:
val_months = ["202310", "202311", "202312"]

val_files = [
    [f"../data/processed/{cfg['sym_a']}_dukascopy_ask_{m}.parquet" for m in val_months],
    [f"../data/processed/{cfg['sym_a']}_dukascopy_bid_{m}.parquet" for m in val_months],
    [f"../data/processed/{cfg['sym_b']}_dukascopy_ask_{m}.parquet" for m in val_months],
    [f"../data/processed/{cfg['sym_b']}_dukascopy_bid_{m}.parquet" for m in val_months],
]

print(f"Building {ACTIVE_PAIR} Validation bars (Oct-Dec 2023)...")
builder_val = SPREAD(agg_type='volume', threshold=1000, active_hours=cfg['hours']) 
df_val = builder_val.build(val_files)
print(f"Built {len(df_val)} volume bars.")

### Run

In [ ]:
print("\nCalculating HMM states on Validation Data (This takes a minute)...")
live_trading_data_val, _ = ENGINE.walk_forward(
    df=df_val, 
    train_days=30,           # Standard real-world rolling history
    coint_window=300,        # Standard beta capture window
    z_window=100,            # Standard mean-reversion window
    k_regimes=2,             # SYNTHETIC RULE: 2 Regimes
    winsorize_std=4.0,       # SYNTHETIC RULE: Cap real-world kurtosis
    scaling=10000,           # SYNTHETIC RULE: Wide scaling for solver stability
    print_freq=20
)
print("Math locked. Ready for Execution Tuning.")

### Optuna

In [ ]:
def objective_val(trial):
    # Execution parameters to tune
    entry_z = trial.suggest_float("entry_z", 1.0, 2.5, step=0.1)
    exit_z = trial.suggest_float("exit_z", -0.5, 0.5, step=0.1)
    danger_threshold = trial.suggest_float("danger_threshold", 0.50, 0.95, step=0.05)
    ar_limit = trial.suggest_float("ar_limit", 0.85, 0.99, step=0.01)

    # Run the lightweight backtester on the pre-calculated HMM data
    bt = BACKTESTER(live_trading_data_val)
    results = bt.run(
        base_z=entry_z, 
        exit_z=exit_z, 
        danger_threshold=danger_threshold, 
        ar_limit=ar_limit,
        fee_bps=0.5, 
        slippage_mode='half_spread'
    )
    
    # Calculate objective (Annualized Sharpe)
    returns = results['Return_MS_AR'].fillna(0)
    
    # Severe penalty for strategies that cause "Paralysis by Analysis"
    if returns.std() == 0 or (results['Target_MS_AR'] != 0).sum() < 20:
        return -99.0 
        
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252 * 24 * 60)
    return sharpe

print("\nHunting for the optimal trading rules...")
study_val = optuna.create_study(direction="maximize", study_name=f"Val_{ACTIVE_PAIR}")
study_val.optimize(objective_val, n_trials=300, show_progress_bar=True)

print(f"\n=== VALIDATION OPTIMIZED PARAMETERS ({ACTIVE_PAIR}) ===")
print(f"Best Validation Sharpe: {study_val.best_value:.2f}")
for key, value in study_val.best_params.items():
    print(f"  {key}: {value:.3f}")

### Plots

In [ ]:
fig1 = ovm.plot_optimization_history(study_val)
fig1.figure.set_size_inches(10, 5)
plt.title(f"{ACTIVE_PAIR}: Execution Tuning History")
plt.tight_layout()
plt.show()

fig2 = ovm.plot_param_importances(study_val)
fig2.figure.set_size_inches(10, 5)
plt.title(f"{ACTIVE_PAIR}: Hyperparameter Importance")
plt.tight_layout()
plt.show()